In [1]:
# Check whether easydiffraction is installed; install it if needed.
# Required for remote environments such as Google Colab.
import importlib.util

if importlib.util.find_spec('easydiffraction') is None:
    %pip install easydiffraction

# Structure Refinement: Co2SiO4, D20 (T-scan)

This example demonstrates a Rietveld refinement of the Co2SiO4 crystal
structure using constant-wavelength neutron powder diffraction data
from D20 at ILL. A sequential refinement is performed against a
temperature scan using sequential fitting, which processes each data
file independently without loading all datasets into memory at once.

## 🛠️ Import Library

In [2]:
import easydiffraction as ed

## 📦 Define Project

The project object manages structures, experiments, analysis, display,
and other related components.

In [3]:
project = ed.Project(name='cosio_d20')
analysis = project.analysis
display = project.display

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

The project must be saved before running sequential fitting, so that
results can be written to `analysis/results.csv`.

In [4]:
project.save_as(dir_path='projects/cosio_d20_scan')

Saving project 📦 'cosio_d20' to '../../../projects/cosio_d20_scan'


├── 📄 project.cif


├── 📁 structures/


├── 📁 experiments/


├── 📁 analysis/


│   └── 📄 analysis.cif


└── 📁 reports/


    └── 📄 cosio_d20.html


## 🧩 Define Structure

This section shows how to add structures and modify their
parameters.

### Create Structure

In [5]:
project.structures.create(name='cosio')
struct = project.structures['cosio']

### Set Space Group

In [6]:
struct.space_group.name_h_m = 'P n m a'
struct.space_group.it_coordinate_system_code = 'abc'

### Set Unit Cell

In [7]:
struct.cell.length_a = 10.31
struct.cell.length_b = 6.0
struct.cell.length_c = 4.79

### Set Atom Sites

In [8]:
struct.atom_sites.create(
    label='Co1',
    type_symbol='Co',
    fract_x=0,
    fract_y=0,
    fract_z=0,
    adp_iso=0.3,
)
struct.atom_sites.create(
    label='Co2',
    type_symbol='Co',
    fract_x=0.279,
    fract_y=0.25,
    fract_z=0.985,
    adp_iso=0.3,
)
struct.atom_sites.create(
    label='Si',
    type_symbol='Si',
    fract_x=0.094,
    fract_y=0.25,
    fract_z=0.429,
    adp_iso=0.34,
)
struct.atom_sites.create(
    label='O1',
    type_symbol='O',
    fract_x=0.091,
    fract_y=0.25,
    fract_z=0.771,
    adp_iso=0.63,
)
struct.atom_sites.create(
    label='O2',
    type_symbol='O',
    fract_x=0.448,
    fract_y=0.25,
    fract_z=0.217,
    adp_iso=0.59,
)
struct.atom_sites.create(
    label='O3',
    type_symbol='O',
    fract_x=0.164,
    fract_y=0.032,
    fract_z=0.28,
    adp_iso=0.83,
)

### Display Structure

In [9]:
project.structure_style.atom_view = 'adp'
project.display.structure(struct_name='cosio')

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

Structure 🧩 'cosio' (Atom view type: 'adp')


## 🔬 Define Experiment

For sequential fitting, we create a single template experiment from
the first data file. This template defines the instrument, peak
profile, background, and linked phases that will be reused for every
data file in the scan.

### Download Data

In [10]:
zip_path = ed.download_data(id=25, destination='data')

Getting data...


Data #25: Co2SiO4, D20 (ILL), 3 files: ~50K, ~300K, ~500K


✅ Data #25 downloaded to '../../../data/ed-25.zip'


### Extract Data Files

In [11]:
scan_data_dir = 'experiments/d20_scan'
data_paths = ed.extract_data_paths_from_zip(
    zip_path,
    destination=project.info.path / scan_data_dir,
)

### Create Template Experiment

In [12]:
project.experiments.add_from_data_path(
    name='d20',
    data_path=data_paths[0],
)
expt = project.experiments['d20']

Data loaded successfully


Experiment 🔬 'd20'. Number of data points: 1507.


### Set Instrument

In [13]:
expt.instrument.setup_wavelength = 1.87
expt.instrument.calib_twotheta_offset = 0.29

### Set Peak Profile

In [14]:
expt.peak.broad_gauss_u = 0.24
expt.peak.broad_gauss_v = -0.53
expt.peak.broad_gauss_w = 0.38
expt.peak.broad_lorentz_y = 0.02

### Set Excluded Regions

In [15]:
expt.excluded_regions.create(id='1', start=0, end=8)
expt.excluded_regions.create(id='2', start=150, end=180)

### Set Background

In [16]:
expt.background.create(id='1', x=8, y=609)
expt.background.create(id='2', x=9, y=581)
expt.background.create(id='3', x=10, y=563)
expt.background.create(id='4', x=11, y=540)
expt.background.create(id='5', x=12, y=520)
expt.background.create(id='6', x=15, y=507)
expt.background.create(id='7', x=25, y=463)
expt.background.create(id='8', x=30, y=434)
expt.background.create(id='9', x=50, y=451)
expt.background.create(id='10', x=70, y=431)
expt.background.create(id='11', x=90, y=414)
expt.background.create(id='12', x=110, y=361)
expt.background.create(id='13', x=130, y=292)
expt.background.create(id='14', x=150, y=241)

### Set Linked Phases

In [17]:
expt.linked_phases.create(id='cosio', scale=1.2)

## 🚀 Perform Analysis

This section shows how to set free parameters, define constraints,
and run the sequential refinement.

### Set Free Parameters

In [18]:
struct.cell.length_a.free = True
struct.cell.length_b.free = True
struct.cell.length_c.free = True

struct.atom_sites['Co2'].fract_x.free = True
struct.atom_sites['Co2'].fract_z.free = True
struct.atom_sites['Si'].fract_x.free = True
struct.atom_sites['Si'].fract_z.free = True
struct.atom_sites['O1'].fract_x.free = True
struct.atom_sites['O1'].fract_z.free = True
struct.atom_sites['O2'].fract_x.free = True
struct.atom_sites['O2'].fract_z.free = True
struct.atom_sites['O3'].fract_x.free = True
struct.atom_sites['O3'].fract_y.free = True
struct.atom_sites['O3'].fract_z.free = True

struct.atom_sites['Co1'].adp_iso.free = True
struct.atom_sites['Co2'].adp_iso.free = True
struct.atom_sites['Si'].adp_iso.free = True
struct.atom_sites['O1'].adp_iso.free = True
struct.atom_sites['O2'].adp_iso.free = True
struct.atom_sites['O3'].adp_iso.free = True

In [19]:
expt.linked_phases['cosio'].scale.free = True

expt.instrument.calib_twotheta_offset.free = True

expt.peak.broad_gauss_u.free = True
expt.peak.broad_gauss_v.free = True
expt.peak.broad_gauss_w.free = True
expt.peak.broad_lorentz_y.free = True

for point in expt.background:
    point.y.free = True

### Set Constraints

Set aliases for parameters.

In [20]:
analysis.aliases.create(
    label='biso_Co1',
    param=struct.atom_sites['Co1'].adp_iso,
)
analysis.aliases.create(
    label='biso_Co2',
    param=struct.atom_sites['Co2'].adp_iso,
)

Set constraints.

In [21]:
analysis.constraints.create(expression='biso_Co2 = biso_Co1')

### Set Minimizer

In [22]:
analysis.minimizer.type = 'bumps (lm)'

Current minimizer changed to


bumps (lm)


### Run Fitting

This is the fitting of the first dataset to optimize the initial
parameters for the sequential fitting. This step is optional but can
help with convergence and speed of the sequential fitting, especially
if the initial parameters are far from optimal.

In [23]:
analysis.fit()

<IPython.core.display.Javascript object>

Standard fitting


📋 Using experiment 🔬 'd20' for 'single' fitting


🚀 Starting fit process with 'bumps (lm)'...


📈 Goodness-of-fit progress:


...

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

└── 📁 reports/


    └── 📄 cosio_d20.html


In [24]:
display.fit.results()

⚙️ Settings used:


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

,Name,Value,Description
1,max_iterations,1000,Maximum solver iterations.


📋 Least-squares fit results:


...

,Metric,Value
1,🧪 Minimizer,bumps (lm)
2,✅ Overall status,success
3,⏱️ Fitting time (seconds),46.61
4,📏 Goodness-of-fit (reduced χ²),4.82
5,"📏 R-factor (Rf, %)",3.16
6,"📏 R-factor squared (Rf², %)",4.68
7,"📏 Weighted R-factor (wR, %)",5.02


📈 Refined parameters:


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

,datablock,category,entry,parameter,units,start,value,s.u.,change
1,cosio,cell,,length_a,Å,10.3100,10.3071,0.0003,0.03 % ↓
2,cosio,cell,,length_b,Å,6.0000,6.0030,0.0002,0.05 % ↑
3,cosio,cell,,length_c,Å,4.7900,4.7865,0.0001,0.07 % ↓
4,cosio,atom_site,Co1,adp_iso,Å²,0.3000,0.1626,0.0807,45.79 % ↓
5,cosio,atom_site,Co2,fract_x,,0.2790,0.2784,0.0007,0.22 % ↓
6,cosio,atom_site,Co2,fract_z,,0.9850,0.9809,0.0015,0.41 % ↓
7,cosio,atom_site,Si,fract_x,,0.0940,0.0934,0.0004,0.61 % ↓
8,cosio,atom_site,Si,fract_z,,0.4290,0.4285,0.0009,0.13 % ↓
9,cosio,atom_site,Si,adp_iso,Å²,0.3400,0.3852,0.0650,13.29 % ↑
10,cosio,atom_site,O1,fract_x,,0.0910,0.0907,0.0003,0.38 % ↓


### Display Correlations

In [25]:
display.fit.correlations()

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

### Display Pattern

In [26]:
display.pattern(expt_name='d20')

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

### Display Structure

In [27]:
project.structure_style.atom_view = 'adp'
project.display.structure(struct_name='cosio')

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

Structure 🧩 'cosio' (Atom view type: 'adp')


### Run Sequential Fitting

Set output verbosity level to "short" to show only one-line status
messages during the analysis process.

In [28]:
project.verbosity = 'short'


Create a persisted extract rule that reads the temperature from each
data file.

In [29]:
temperature = 'diffrn.ambient_temperature'

In [30]:
analysis.sequential_fit_extract.create(
    id='temperature',
    target=temperature,
    pattern=r'^TEMP\s+([0-9.]+)',
    required=True,
)

Set the sequential fitting parameters.

In [31]:
analysis.fitting_mode.type = 'sequential'
analysis.sequential_fit.data_dir = scan_data_dir
analysis.sequential_fit.max_workers = 'auto'
analysis.sequential_fit.reverse = True

Fitting mode changed to


sequential


Run the sequential fit over all data files in the scan directory.

In [32]:
analysis.fit()

<IPython.core.display.Javascript object>

Sequential fitting


🚀 Starting fit process with 'bumps (lm)'...


📋 3 files in 1 chunks (max_workers=4)


📈 Goodness-of-fit progress:


...

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

└── 📁 reports/


    └── 📄 cosio_d20.html


### Replay a Dataset

Apply fitted parameters from the first CSV row and plot the result.

In [33]:
project.apply_params_from_csv(row_index=0)
display.pattern(expt_name='d20')

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>


Apply fitted parameters from the last CSV row and plot the result.

In [34]:
project.apply_params_from_csv(row_index=-1)
display.pattern(expt_name='d20')

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

### Display Parameter Evolution

Reuse the extracted diffrn path as the x-axis in the following plots.

Plot fit quality metrics vs. temperature.

In [35]:
display.fit.series(analysis.fit_result.success, versus=temperature)
display.fit.series(analysis.fit_result.reduced_chi_square, versus=temperature)
display.fit.series(analysis.fit_result.iterations, versus=temperature)

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

...

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

Plot unit cell parameters vs. temperature.

In [36]:
display.fit.series(struct.cell.length_a, versus=temperature)
display.fit.series(struct.cell.length_b, versus=temperature)
display.fit.series(struct.cell.length_c, versus=temperature)

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

...

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

Plot isotropic displacement parameters vs. temperature.

In [37]:
display.fit.series(struct.atom_sites['Co1'].adp_iso, versus=temperature)
display.fit.series(struct.atom_sites['Si'].adp_iso, versus=temperature)
display.fit.series(struct.atom_sites['O1'].adp_iso, versus=temperature)
display.fit.series(struct.atom_sites['O2'].adp_iso, versus=temperature)
display.fit.series(struct.atom_sites['O3'].adp_iso, versus=temperature)

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

...

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

Plot selected fractional coordinates vs. temperature.

In [38]:
display.fit.series(struct.atom_sites['Co2'].fract_x, versus=temperature)
display.fit.series(struct.atom_sites['Co2'].fract_z, versus=temperature)
display.fit.series(struct.atom_sites['O1'].fract_z, versus=temperature)
display.fit.series(struct.atom_sites['O2'].fract_z, versus=temperature)
display.fit.series(struct.atom_sites['O3'].fract_z, versus=temperature)

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

...

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>